In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

SEED = 42

In [3]:
iris = load_iris()
X = iris.data
y = iris.target

flowers = pd.DataFrame(X, columns=iris.feature_names)
flowers["species"] = iris.target_names[y]

print("First five flowers:")
print(flowers.head())
print("\nTotal flowers:", len(X))
print("Missing measurements:", np.isnan(X).sum())
print("Species:", iris.target_names)

First five flowers:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

  species  
0  setosa  
1  setosa  
2  setosa  
3  setosa  
4  setosa  

Total flowers: 150
Missing measurements: 0
Species: ['setosa' 'versicolor' 'virginica']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

print("\nTraining flowers:", len(X_train))
print("Testing flowers:", len(X_test))


Training flowers: 120
Testing flowers: 30


In [6]:
def evaluate_predictions(actual, predicted):
    accuracy = accuracy_score(actual, predicted)
    f1 = f1_score(
        actual, predicted,
        labels=[0, 1, 2],
        average="macro",
        zero_division=0
    )
    return accuracy, f1

In [7]:
rng = np.random.default_rng(SEED)
number_of_flowers = len(X_train)
all_indices = np.arange(number_of_flowers)

sample_indices = rng.choice(
    all_indices,
    size=number_of_flowers,
    replace=True
)

unique_indices = np.unique(sample_indices)
oob_indices = np.setdiff1d(all_indices, unique_indices)

print("\nBOOTSTRAPPING")
print("Total draws:", len(sample_indices))
print("Different flowers selected:", len(unique_indices))
print("Out-of-bag flowers:", len(oob_indices))


BOOTSTRAPPING
Total draws: 120
Different flowers selected: 79
Out-of-bag flowers: 41


In [8]:
rng = np.random.default_rng(SEED)
bootstrap_accuracy = []
bootstrap_f1 = []

for repetition in range(100):
    sample_indices = rng.choice(
        all_indices,
        size=number_of_flowers,
        replace=True
    )

    oob_indices = np.setdiff1d(
        all_indices,
        np.unique(sample_indices)
    )

    if len(oob_indices) == 0:
        continue

    model = DecisionTreeClassifier(
        max_depth=3,
        random_state=SEED
    )

    model.fit(
        X_train[sample_indices],
        y_train[sample_indices]
    )

    predictions = model.predict(X_train[oob_indices])

    accuracy, f1 = evaluate_predictions(
        y_train[oob_indices], predictions
    )

    bootstrap_accuracy.append(accuracy)
    bootstrap_f1.append(f1)

print("\nBootstrap repetitions:", len(bootstrap_accuracy))
print("Mean accuracy:", round(np.mean(bootstrap_accuracy), 4))
print("Mean macro F1:", round(np.mean(bootstrap_f1), 4))




Bootstrap repetitions: 100
Mean accuracy: 0.9441
Mean macro F1: 0.9426


In [9]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

cv_accuracy = []
cv_f1 = []

for fold, (fit_indices, validation_indices) in enumerate(
    cv.split(X_train, y_train), start=1
):
    model = DecisionTreeClassifier(
        max_depth=3,
        random_state=SEED
    )

    model.fit(
        X_train[fit_indices],
        y_train[fit_indices]
    )

    predictions = model.predict(X_train[validation_indices])

    accuracy, f1 = evaluate_predictions(
        y_train[validation_indices], predictions
    )

    cv_accuracy.append(accuracy)
    cv_f1.append(f1)

    print(
        f"Fold {fold}: Accuracy = {accuracy:.4f}, "
        f"Macro F1 = {f1:.4f}"
    )

Fold 1: Accuracy = 0.9583, Macro F1 = 0.9582
Fold 2: Accuracy = 1.0000, Macro F1 = 1.0000
Fold 3: Accuracy = 0.9583, Macro F1 = 0.9582
Fold 4: Accuracy = 0.9583, Macro F1 = 0.9582
Fold 5: Accuracy = 0.9167, Macro F1 = 0.9153
